# FAISS Vector Retrieval + Hybrid Graph Notebook

This notebook runs retrieval after `index.faiss` and `payloads.jsonl` are available, and (when v2 graph sources are present) demonstrates the **primary hybrid pipeline**:

```text
query → embed → vector seed retrieval → graph expansion + validity/authority overlays → optional LLM generation
```

**Architecture**
- Store: [`SQLitePayloadFaissVectorStore`](../src/retrieval/sqlite_faiss_store.py) — FAISS + rebuild-if-stale `payload_cache.sqlite`
- Retrieval: [`VectorRetriever`](../src/retrieval/retriever.py) — seed search; optional graph expansion / graph-guided filter
- Knowledge graph: [`KnowledgeGraphFacade`](../src/knowledge_graph/facade.py), [`GraphExpansion`](../src/knowledge_graph/expansion.py), overlays
- Generation: [`generation.reasoning_client`](../src/generation/reasoning_client.py) — OpenAI-compatible client with three-way reasoning parse

**Primary vs secondary graph paths**
- **Primary (default full pipeline):** vector-first hybrid expansion (seed hits → graph expand → overlays → generate)
- **Secondary (optional demo):** graph-guided pre-filter (document whitelist before vector search) — not the default `ask()` path

Expected layouts:

```text
data/faiss_index/
  index.faiss
  payloads.jsonl
  id_map.json        # optional
  payload_cache.sqlite  # built automatically on first load

data/v2/             # required for hybrid graph path
  documents.jsonl, provisions.jsonl, chunks.jsonl, edges.jsonl, external_stubs.jsonl
  validity_timeline.jsonl, authority_index.jsonl   # optional overlays
```

Run cells top to bottom. Pure vector profiles (`current_law`, `broad`, `historical`) remain usable if the graph is missing. Hybrid mode fails clearly when the graph is unavailable rather than silently falling back under a hybrid label.

**Note:** This notebook is a hybrid **demonstration** layered on existing modules — not a replacement for dedicated graph verification ([`scripts/verify_kg.py`](../scripts/verify_kg.py)) or judged evaluation ([`scripts/evaluate_e2e.py`](../scripts/evaluate_e2e.py)).

---

### Colab RAM notes (hybrid retained)

Peak RSS on free Colab is dominated by three residents:

1. **FAISS** — exact `IndexFlatIP` is ~6 GB for ~1.5M×1024-d float32. Prefer a prebuilt **IVFPQ** `index.faiss` (rebuild once with [`scripts/rebuild_faiss_ivfpq.py`](../scripts/rebuild_faiss_ivfpq.py); payloads/SQLite unchanged).
2. **Embedder** — `multilingual-e5-large` is ~2–3 GB. This notebook uses a **lazy** embedder (`LAZY_EMBEDDER=True`) so the model loads on first query, not at store load.
3. **Knowledge graph** — structural `.gpickle` is multi-GB. Hybrid stays the default, but `LAZY_GRAPH_LOAD=True` defers gpickle + overlay join until the first hybrid call.

Use the memory probe cells (`print_memory`) after FAISS / embedder / graph to verify budgets. Hybrid still fails clearly if the graph is missing when hybrid is requested.


## 1. Environment setup

In [1]:
!git clone https://github.com/PhuongThao-2005/TextMining.git

Cloning into 'TextMining'...
remote: Enumerating objects: 336, done.
remote: Counting objects: 100% (336/336), done.
remote: Compressing objects: 100% (267/267), done.
remote: Total 336 (delta 101), reused 286 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (336/336), 528.91 KiB | 2.13 MiB/s, done.
Resolving deltas: 100% (101/101), done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Optional: install runtime dependencies if your environment does not have them yet.
# Uncomment and run once if needed.
%pip install -q faiss-cpu sentence-transformers pandas openai psutil


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 138.1 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import json
import os
import sys
import time
import logging

PROJECT_ROOT = Path("/content/TextMining")
if not (PROJECT_ROOT / 'src').exists():
    # Useful if the notebook is launched from notebooks/.
    PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print('HF Hub token detected in environment.')
else:
    logging.getLogger('huggingface_hub.utils._http').setLevel(logging.ERROR)
    print('HF_TOKEN not set; suppressing the Hugging Face unauthenticated-request warning.')

print('Project root:', PROJECT_ROOT)
print('src on path:', SRC_DIR.exists())


HF_TOKEN not set; suppressing the Hugging Face unauthenticated-request warning.
Project root: /content/TextMining
src on path: True


## 2. Configure artifact paths and retrieval settings

Change `INDEX_DIR` if you download the FAISS files somewhere else.

In [5]:
# Directory containing index.faiss + payloads.jsonl (+ optional id_map.json)
INDEX_DIR = Path("/content/drive/Shareddrives/[Text Mining] - Project/faiss_index")
# Graph + overlay sources (structural graph under data/v2)
V2_DATA_DIR = Path("/content/drive/Shareddrives/[Text Mining] - Project/Data/pre-processed")

# Must match the embedding model used to build index.faiss.
# Alias EMBEDDING_MODEL_NAME kept for plan/quickstart naming parity.
EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'
EMBEDDING_MODEL_NAME = EMBEDDING_MODEL

TOP_K = 30                 # candidates pulled from FAISS before reranking/dedup
TOP_N = 10                 # final chunks returned
SCORE_THRESHOLD = 0.30
# Local same-provision expansion (payload-window). Prefer False when demoing graph expansion.
EXPAND_UNITS = False
LOCAL_EXPAND_UNITS = EXPAND_UNITS  # alias: local_expand_units mechanism label
DEFAULT_FILTER_PROFILE = 'broad'  # current_law | broad | historical (non-graph)
FILTER_PROFILE = DEFAULT_FILTER_PROFILE
BENCHMARK_SAMPLE_SIZE = 10

# --- Hybrid graph settings (003-notebook-graph-integration / FR-021) ---
ENABLE_HYBRID_EXPANSION = True
HYBRID_MAX_HOP = 1
HYBRID_MAX_CONTEXT = 12
AS_OF_DATE = '2026-07-13'
USE_HYBRID_EVIDENCE_FOR_GENERATION = True
ENABLE_GRAPH_GUIDED_PREFILTER_DEMO = False  # secondary whitelist-before-search path
GRAPH_GUIDED_START_ID = ''  # optional document id_str; empty → take from first seed hit
GRAPH_GUIDED_TRAVERSAL_MODE = 'basis'
GRAPH_GUIDED_MAX_DEPTH = 3

# --- RAM / Colab controls (hybrid retained) ---
# Lazy embedder: defer multilingual-e5-large (~2-3GB) until first encode.
LAZY_EMBEDDER = True
# Lazy graph: defer gpickle + overlay join until first hybrid call.
# Hybrid remains the default pipeline; first hybrid request pays the load cost.
LAZY_GRAPH_LOAD = True
# IVF search probes when index.faiss is IndexIVFPQ (ignored for Flat).
FAISS_NPROBE = 32
# Optional heavy exports (not present in this notebook; keep False if added).
RUN_EXPORTS = False

print('INDEX_DIR:', INDEX_DIR)
print('V2_DATA_DIR:', V2_DATA_DIR)
print('ENABLE_HYBRID_EXPANSION:', ENABLE_HYBRID_EXPANSION)
print('USE_HYBRID_EVIDENCE_FOR_GENERATION:', USE_HYBRID_EVIDENCE_FOR_GENERATION)
print('LOCAL_EXPAND_UNITS / EXPAND_UNITS:', LOCAL_EXPAND_UNITS)
print('ENABLE_GRAPH_GUIDED_PREFILTER_DEMO:', ENABLE_GRAPH_GUIDED_PREFILTER_DEMO)
print('LAZY_EMBEDDER:', LAZY_EMBEDDER)
print('LAZY_GRAPH_LOAD:', LAZY_GRAPH_LOAD)
print('FAISS_NPROBE:', FAISS_NPROBE)
print('RUN_EXPORTS:', RUN_EXPORTS)
INDEX_DIR


INDEX_DIR: /content/drive/Shareddrives/[Text Mining] - Project/faiss_index
V2_DATA_DIR: /content/drive/Shareddrives/[Text Mining] - Project/Data/pre-processed
ENABLE_HYBRID_EXPANSION: True
USE_HYBRID_EVIDENCE_FOR_GENERATION: True
LOCAL_EXPAND_UNITS / EXPAND_UNITS: False
ENABLE_GRAPH_GUIDED_PREFILTER_DEMO: False


PosixPath('/content/drive/Shareddrives/[Text Mining] - Project/faiss_index')

## 3. Preflight: wait until downloads are complete

In [6]:
phase_t0 = time.perf_counter()
required_files = [INDEX_DIR / 'index.faiss', INDEX_DIR / 'payloads.jsonl']
optional_files = [INDEX_DIR / 'id_map.json']

missing = [p for p in required_files if not p.exists()]
if missing:
    print('Downloads are not ready yet. Missing:')
    for p in missing:
        print(' -', p)
else:
    print('Required files found.')
    for p in required_files + optional_files:
        if p.exists():
            print(f'{p.name}: {p.stat().st_size / 1024 / 1024:.2f} MB')
        else:
            print(f'{p.name}: not found (optional)')

print(f'Preflight completed in {time.perf_counter() - phase_t0:.2f}s')


Required files found.
index.faiss: 5911.63 MB
payloads.jsonl: 4836.18 MB
id_map.json: 60.91 MB
Preflight completed in 0.43s


## 4. Load the FAISS store and build retriever

In [7]:
if missing:
    raise FileNotFoundError('Download index.faiss and payloads.jsonl before running this cell.')

from retrieval.config import VectorIndexConfig
from retrieval.embeddings import LazySentenceTransformerEmbedder, SentenceTransformerEmbedder
from retrieval.memory_utils import print_memory
from retrieval.retriever import VectorRetriever
from retrieval.sqlite_faiss_store import SQLitePayloadFaissVectorStore

load_t0 = time.perf_counter()
config = VectorIndexConfig(
    embedding_model=EMBEDDING_MODEL,
    top_k=TOP_K,
    top_n=TOP_N,
    score_threshold=SCORE_THRESHOLD,
    expand_units=EXPAND_UNITS,
)

print_memory('before_faiss_load')
store = SQLitePayloadFaissVectorStore.load(INDEX_DIR, nprobe=FAISS_NPROBE)
print_memory('after_faiss_load')

if LAZY_EMBEDDER:
    embedder = LazySentenceTransformerEmbedder(
        EMBEDDING_MODEL,
        query_prefix=config.query_prefix,
        passage_prefix=config.passage_prefix,
        expected_dimension=1024,
    )
    print('Embedder: LazySentenceTransformerEmbedder (loads on first query)')
else:
    embedder = SentenceTransformerEmbedder(
        EMBEDDING_MODEL,
        query_prefix=config.query_prefix,
        passage_prefix=config.passage_prefix,
    )
    print_memory('after_embedder_load')

retriever = VectorRetriever(config=config, embedder=embedder, store=store)

print(f'Vector retriever ready in {time.perf_counter() - load_t0:.2f}s')
print(f'Loaded FAISS vectors: {store.total_vectors:,}')
print(f'Embedding dimension (declared): {embedder.dimension}')
print('Store class:', type(store).__module__ + '.' + type(store).__name__)
print('FAISS class:', type(store.index).__name__)
if hasattr(store.index, 'nprobe'):
    print('FAISS nprobe:', store.index.nprobe)

# Vector-only retriever (graph_expansion=None). Hybrid retriever is wired after graph load.
hybrid_retriever = None
graph_expansion = None


Reusing SQLite payload cache: payload_cache.sqlite
FAISS index loaded in 87.34s


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Vector retriever ready in 138.06s
Loaded FAISS vectors: 1,513,376
Embedding dimension: 1024
Store class: retrieval.sqlite_faiss_store.SQLitePayloadFaissVectorStore


/content/TextMining/src/retrieval/embeddings.py:54: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self.model.get_sentence_embedding_dimension()


## 4.3 Hybrid graph integration (vector-first)

Primary hybrid path (default full pipeline):

```text
query → embed → vector seed retrieve → resolve chunk→provision→document
      → GraphExpansion + validity/authority overlays → fused evidence → optional LLM
```

Secondary path (optional only): graph-guided pre-filter whitelist before vector search.

This section orchestrates existing modules under `src/knowledge_graph/` and `src/retrieval/` — it does **not** reimplement graph logic and is **not** a replacement for `scripts/verify_kg.py` or `scripts/evaluate_e2e.py`.


In [8]:
KG_PICKLE_PATH = Path("/content/drive/Shareddrives/[Text Mining] - Project/KG/knowledge_graph.gpickle")
# Keep False for normal notebook use: load pickle only (no JSONL rebuild).
ALLOW_GRAPH_JSONL_REBUILD = False

In [9]:
# ### HYBRID_GRAPH_INTEGRATION — intended import surface (FR-002)
from dataclasses import dataclass, field
from typing import Any

from knowledge_graph import (
    GraphBuildStats,
    GraphExpansion,
    GraphLoaderPaths,
    GraphPickleCorruptError,
    GraphPickleIncompatibleError,
    GraphPickleNotFoundError,
    KnowledgeGraphFacade,
    QueryConstraints,
    load_knowledge_graph,
    parse_authority_index_rows,
    parse_validity_event_rows,
)
from knowledge_graph.context_schema import GraphGuidedFilter
from knowledge_graph.expansion_schema import ExpansionResult
from knowledge_graph.overlay import OverlayBundle
from knowledge_graph.overlay_schema import DocumentOverlay
from retrieval.io_utils import read_jsonl
from retrieval.schema import RetrievedChunk, RetrievalResult
from retrieval.stores import SearchHit

print('Hybrid import surface ready: KnowledgeGraphFacade, GraphExpansion, overlays, GraphGuidedFilter')


Hybrid import surface ready: KnowledgeGraphFacade, GraphExpansion, overlays, GraphGuidedFilter


In [ ]:
# ### HYBRID_GRAPH_INTEGRATION - preflight, gpickle load (preferred), overlays, expansion wire, guard
# RAM: when LAZY_GRAPH_LOAD=True, structural gpickle + overlays load on first hybrid call.


@dataclass
class GraphLoadStatus:
    structural_ready: bool = False
    overlays_ready: bool = False
    missing_structural_files: list[str] = field(default_factory=list)
    missing_overlay_files: list[str] = field(default_factory=list)
    pickle_path: str | None = None
    load_source: str | None = None  # 'gpickle' | 'jsonl_rebuild'
    format_version: int | None = None
    created_at_utc: str | None = None
    source_data_dir: str | None = None
    build_stats: Any = None
    build_warnings: tuple[str, ...] = ()
    as_of_date: str | None = None
    overlay_coverage: dict[str, int] | None = None
    error: str | None = None
    build_duration_s: float | None = None
    loaded: bool = False  # True after ensure_hybrid_graph() actually materializes kg_graph


def preflight_graph_sources(v2_dir: Path, pickle_path: Path) -> GraphLoadStatus:
    """Preflight pickle + optional overlay/JSONL sources.

    Structural readiness for normal notebook use is driven by the portable
    `.gpickle` artifact. JSONL structural files are only required when
    ALLOW_GRAPH_JSONL_REBUILD is True and the pickle is missing.
    """
    paths = GraphLoaderPaths(data_dir=v2_dir)
    missing_structural = [str(p) for p in paths.required_paths() if not p.exists()]
    overlay_names = ('validity_timeline.jsonl', 'authority_index.jsonl')
    missing_overlay = [str(v2_dir / name) for name in overlay_names if not (v2_dir / name).exists()]
    pickle_ok = pickle_path.is_file()

    # Prefer pickle; do not require JSONL for structural_ready in default mode.
    if pickle_ok:
        structural_ready = True
    elif ALLOW_GRAPH_JSONL_REBUILD and not missing_structural:
        structural_ready = True
    else:
        structural_ready = False

    status = GraphLoadStatus(
        structural_ready=structural_ready,
        overlays_ready=not missing_overlay,
        missing_structural_files=missing_structural,
        missing_overlay_files=missing_overlay,
        pickle_path=str(pickle_path),
    )

    print('=== Graph preflight ===')
    print('V2_DATA_DIR:', v2_dir)
    print('KG_PICKLE_PATH:', pickle_path)
    print(f'  [{"OK" if pickle_ok else "MISSING"}] {pickle_path}')
    print('ALLOW_GRAPH_JSONL_REBUILD:', ALLOW_GRAPH_JSONL_REBUILD)
    print('LAZY_GRAPH_LOAD:', LAZY_GRAPH_LOAD)
    print('Structural JSONL files (only needed for rebuild fallback):')
    for p in paths.required_paths():
        flag = 'OK' if p.exists() else 'MISSING'
        print(f'  [{flag}] {p}')
    print('Overlay files (optional, still from JSONL after structural load):')
    for name in overlay_names:
        p = v2_dir / name
        flag = 'OK' if p.exists() else 'MISSING'
        print(f'  [{flag}] {p}')

    if not pickle_ok:
        print('Pickle MISSING - notebook will not rebuild from JSONL unless ALLOW_GRAPH_JSONL_REBUILD=True.')
        print('  Build once with: python scripts/build_kg_pickle.py --data-dir data/v2 --output data/graph/knowledge_graph.gpickle')
    if missing_overlay:
        print('Overlays unavailable (currency/authority labeled unavailable if structural graph loads).')
        for m in missing_overlay:
            print(' -', m)
    if not structural_ready:
        print('Structural graph UNAVAILABLE. Pure vector profiles remain usable.')
        if not pickle_ok:
            print(' - missing pickle:', pickle_path)
        if ALLOW_GRAPH_JSONL_REBUILD and missing_structural:
            print('Missing structural JSONL files for rebuild:')
            for m in missing_structural:
                print(' -', m)
    return status


def _stat_value(stats: Any, name: str, default: Any = 'n/a') -> Any:
    if stats is None:
        return default
    if isinstance(stats, dict):
        return stats.get(name, default)
    return getattr(stats, name, default)


def print_graph_stats(stats: Any, warnings: tuple[str, ...] | list[str] = (), *, title: str = 'Graph Statistics') -> None:
    print(f'[{title}]')
    print(f'  documents:           {_stat_value(stats, "document_count")}')
    print(f'  external_stubs:      {_stat_value(stats, "external_stub_count")}')
    print(f'  provisions:          {_stat_value(stats, "provision_count")}')
    print(f'  chunks:              {_stat_value(stats, "chunk_count")}')
    print(f'  document_edges:      {_stat_value(stats, "document_edge_count")}')
    print(f'  verified_edges:      {_stat_value(stats, "verified_document_edge_count")}')
    print(f'  unverified_edges:    {_stat_value(stats, "unverified_document_edge_count")}')
    print(f'  structural_edges:    {_stat_value(stats, "structural_edge_count")}')
    orphan_p = _stat_value(stats, 'orphan_provision_count', None)
    orphan_c = _stat_value(stats, 'orphan_chunk_count', None)
    if orphan_p is not None:
        print(f'  orphan_provisions:   {orphan_p}')
    if orphan_c is not None:
        print(f'  orphan_chunks:       {orphan_c}')
    if warnings:
        print('Warnings:')
        for w in list(warnings)[:20]:
            print(' -', w)


def load_structural_graph_from_pickle(pickle_path: Path):
    """Load structural KnowledgeGraph from trusted project .gpickle (no JSONL rebuild)."""
    load_t0 = time.perf_counter()
    loaded = load_knowledge_graph(pickle_path)
    duration = time.perf_counter() - load_t0
    return loaded, duration


def rebuild_structural_graph_from_jsonl(v2_dir: Path):
    """Fallback only: parse v2 JSONL and build in-memory graph."""
    kg_paths = GraphLoaderPaths(data_dir=v2_dir)
    facade = KnowledgeGraphFacade(paths=kg_paths)
    build_t0 = time.perf_counter()
    build_result = facade.build_graph()
    duration = time.perf_counter() - build_t0
    return facade, build_result, duration


def _join_overlays_streaming(kg_facade, kg_graph, v2_dir: Path, as_of: str):
    """Join overlays without holding both raw JSONL lists and the bundle longer than needed.

    Streams parse generators into the joiner; still materializes the final
    OverlayBundle (required for O(1) per-document lookup during hybrid).
    """
    validity_path = v2_dir / 'validity_timeline.jsonl'
    authority_path = v2_dir / 'authority_index.jsonl'
    # Generators — OverlayJoiner.index_* materializes internally once.
    validity_events = parse_validity_event_rows(read_jsonl(validity_path))
    authority_entries = parse_authority_index_rows(read_jsonl(authority_path))
    bundle = kg_facade.build_overlay_bundle(
        documents=kg_graph.documents.values(),
        validity_events=validity_events,
        authority_entries=authority_entries,
        as_of_date=as_of,
    )
    return bundle


def materialize_hybrid_graph() -> None:
    """Actually load gpickle + overlays + wire hybrid_retriever (idempotent)."""
    global kg_facade, kg_graph, kg_build_result, kg_load_result
    global overlay_bundle, graph_expansion, hybrid_retriever, document_overlays
    global graph_load_status

    if graph_load_status.loaded and kg_graph is not None and graph_expansion is not None:
        return

    if not graph_load_status.structural_ready:
        raise RuntimeError(
            'Cannot materialize hybrid graph: structural sources unavailable. '
            f'Detail: {graph_load_status.error or graph_load_status.pickle_path}'
        )

    try:
        from retrieval.memory_utils import print_memory

        print_memory('before_graph_load')
        pickle_path = Path(graph_load_status.pickle_path or KG_PICKLE_PATH)
        if pickle_path.is_file():
            print('\n=== Graph load (gpickle) ===')
            print('Using portable structural pickle - not rebuilding from JSONL.')
            kg_load_result, duration = load_structural_graph_from_pickle(pickle_path)
            kg_graph = kg_load_result.graph
            stats = kg_load_result.stats
            warnings = tuple(kg_load_result.warnings or ())
            kg_facade = KnowledgeGraphFacade(paths=GraphLoaderPaths(data_dir=V2_DATA_DIR))
            graph_load_status.load_source = 'gpickle'
            graph_load_status.format_version = kg_load_result.format_version
            graph_load_status.created_at_utc = kg_load_result.created_at_utc
            graph_load_status.source_data_dir = kg_load_result.source_data_dir
            graph_load_status.build_stats = stats
            graph_load_status.build_warnings = warnings
            graph_load_status.build_duration_s = duration
            graph_load_status.structural_ready = True
            print(f'Knowledge graph loaded from gpickle in {duration:.2f}s')
            print(f'  path: {kg_load_result.path}')
            print(f'  format_version: {kg_load_result.format_version}')
            print(f'  created_at_utc: {kg_load_result.created_at_utc}')
            print(f'  source_data_dir: {kg_load_result.source_data_dir}')
            if stats is None:
                stats = GraphBuildStats(
                    document_count=len(kg_graph.documents),
                    external_stub_count=len(kg_graph.external_stubs),
                    provision_count=len(kg_graph.provisions),
                    chunk_count=len(kg_graph.chunks),
                    document_edge_count=len(kg_graph.document_edges),
                    verified_document_edge_count=len(kg_graph.verified_document_edges),
                    unverified_document_edge_count=(
                        len(kg_graph.document_edges) - len(kg_graph.verified_document_edges)
                    ),
                    structural_edge_count=len(kg_graph.structural_edges),
                    orphan_provision_count=0,
                    orphan_chunk_count=0,
                    missing_external_stub_count=0,
                    structural_edge_counts={},
                    edge_group_counts={},
                )
                graph_load_status.build_stats = stats
            print_graph_stats(stats, warnings, title='Loaded Graph Statistics')
        elif ALLOW_GRAPH_JSONL_REBUILD:
            print('\n=== Graph build (JSONL rebuild fallback) ===')
            print('Pickle missing and ALLOW_GRAPH_JSONL_REBUILD=True - building from structural JSONL.')
            kg_facade, kg_build_result, duration = rebuild_structural_graph_from_jsonl(V2_DATA_DIR)
            kg_graph = kg_build_result.graph
            stats = kg_build_result.stats
            warnings = tuple(kg_build_result.warnings or ())
            graph_load_status.load_source = 'jsonl_rebuild'
            graph_load_status.build_stats = stats
            graph_load_status.build_warnings = warnings
            graph_load_status.build_duration_s = duration
            graph_load_status.structural_ready = True
            print(f'Knowledge graph built from JSONL in {duration:.2f}s')
            print_graph_stats(stats, warnings, title='Build Statistics')
        else:
            raise RuntimeError(
                f'Structural graph pickle not found at {pickle_path}. '
                'Build it with scripts/build_kg_pickle.py or set ALLOW_GRAPH_JSONL_REBUILD=True.'
            )

        # Overlays remain dynamic/optional after structural load (not in pickle).
        if not graph_load_status.missing_overlay_files:
            print('\n=== Overlay join ===')
            if kg_facade is None:
                kg_facade = KnowledgeGraphFacade(paths=GraphLoaderPaths(data_dir=V2_DATA_DIR))
            overlay_bundle = _join_overlays_streaming(kg_facade, kg_graph, V2_DATA_DIR, AS_OF_DATE)
            document_overlays = dict(overlay_bundle.document_overlays)
            graph_load_status.overlays_ready = True
            graph_load_status.as_of_date = AS_OF_DATE
            currency_hist: dict[str, int] = {}
            for ov in document_overlays.values():
                currency_hist[ov.currency_status] = currency_hist.get(ov.currency_status, 0) + 1
            # Avoid second full materialization of validity/authority lists for counts.
            graph_load_status.overlay_coverage = {
                'docs_with_overlay': len(document_overlays),
                **{f'currency_{k}': v for k, v in sorted(currency_hist.items())},
            }
            print(f'Overlay bundle for as_of_date={AS_OF_DATE}')
            print(f'  docs_with_overlay: {len(document_overlays):,}')
            print('  currency histogram (sample keys):', dict(list(currency_hist.items())[:8]))
        else:
            graph_load_status.overlays_ready = False
            print('\nOverlays MISSING - structural expansion allowed; currency/authority labeled unavailable.')

        graph_expansion = GraphExpansion(kg_graph)
        hybrid_retriever = VectorRetriever(
            config=config,
            embedder=embedder,
            store=store,
            graph_expansion=graph_expansion,
        )
        graph_load_status.loaded = True
        graph_load_status.error = None
        print_memory('after_graph_load')
        print('\nGraphExpansion wired. hybrid_retriever has graph_expansion; vector-only retriever remains graph_expansion=None.')
        print('Label reminder: graph_expansion ≠ local_expand_units')
        print('load_source:', graph_load_status.load_source)
    except (GraphPickleNotFoundError, GraphPickleCorruptError, GraphPickleIncompatibleError) as exc:
        graph_load_status.structural_ready = False
        graph_load_status.loaded = False
        graph_load_status.error = str(exc)
        kg_facade = None
        kg_graph = None
        graph_expansion = None
        hybrid_retriever = None
        print('Graph pickle load FAILED:', exc)
        print('Pure vector retrieval remains usable; hybrid mode will fail clearly if requested.')
        print('Build/fix pickle with: python scripts/build_kg_pickle.py --data-dir data/v2 --output data/graph/knowledge_graph.gpickle')
        raise
    except Exception as exc:
        graph_load_status.structural_ready = False
        graph_load_status.loaded = False
        graph_load_status.error = str(exc)
        kg_facade = None
        kg_graph = None
        graph_expansion = None
        hybrid_retriever = None
        print('Graph load/build FAILED:', exc)
        print('Pure vector retrieval remains usable; hybrid mode will fail clearly if requested.')
        raise


def ensure_hybrid_graph() -> None:
    """Ensure graph is loaded when hybrid is needed (lazy or eager)."""
    if graph_load_status.loaded and kg_graph is not None and graph_expansion is not None:
        return
    materialize_hybrid_graph()


graph_load_status = preflight_graph_sources(V2_DATA_DIR, KG_PICKLE_PATH)
kg_facade: KnowledgeGraphFacade | None = None
kg_graph = None
kg_build_result = None
kg_load_result = None
overlay_bundle: OverlayBundle | None = None
graph_expansion: GraphExpansion | None = None
hybrid_retriever: VectorRetriever | None = None
document_overlays: dict[str, DocumentOverlay] = {}

if graph_load_status.structural_ready and not LAZY_GRAPH_LOAD:
    print('LAZY_GRAPH_LOAD=False — loading hybrid graph eagerly.')
    try:
        materialize_hybrid_graph()
    except Exception:
        pass  # errors already recorded on graph_load_status
elif graph_load_status.structural_ready and LAZY_GRAPH_LOAD:
    print('LAZY_GRAPH_LOAD=True — deferring gpickle/overlay load until first hybrid call.')
    print('Hybrid remains the default pipeline; ensure_hybrid_graph() runs inside hybrid helpers.')
else:
    print('Skipping graph load (pickle missing and JSONL rebuild disabled or unavailable).')


def require_graph_for_hybrid(action: str = 'hybrid expansion') -> None:
    """Fail clearly if hybrid is requested without a loaded graph (FR-015)."""
    # Lazy path: attempt load once when hybrid is actually requested.
    if (
        graph_load_status.structural_ready
        and (not graph_load_status.loaded or kg_graph is None or graph_expansion is None)
    ):
        try:
            ensure_hybrid_graph()
        except Exception as exc:
            # ensure_hybrid_graph already stamped graph_load_status.error
            pass

    if not graph_load_status.structural_ready or kg_graph is None or graph_expansion is None:
        missing = []
        if graph_load_status.pickle_path and not Path(graph_load_status.pickle_path).is_file():
            missing.append(f'missing pickle: {graph_load_status.pickle_path}')
        if graph_load_status.missing_structural_files:
            missing.extend(graph_load_status.missing_structural_files)
        if not missing:
            missing = ['(structural graph not loaded)']
        detail = graph_load_status.error or '; '.join(missing)
        raise RuntimeError(
            f"Cannot run {action}: knowledge graph unavailable. "
            f"Do not silently fall back to vector-only under a hybrid label. Detail: {detail}"
        )


print('\nGraphLoadStatus:')
print('  structural_ready:', graph_load_status.structural_ready)
print('  loaded:', graph_load_status.loaded)
print('  load_source:', graph_load_status.load_source)
print('  pickle_path:', graph_load_status.pickle_path)
print('  overlays_ready:', graph_load_status.overlays_ready)
print('  error:', graph_load_status.error)


=== Graph preflight ===
V2_DATA_DIR: /content/drive/Shareddrives/[Text Mining] - Project/Data/pre-processed
KG_PICKLE_PATH: /content/drive/Shareddrives/[Text Mining] - Project/KG/knowledge_graph.gpickle
  [OK] /content/drive/Shareddrives/[Text Mining] - Project/KG/knowledge_graph.gpickle
ALLOW_GRAPH_JSONL_REBUILD: False
Structural JSONL files (only needed for rebuild fallback):
  [OK] /content/drive/Shareddrives/[Text Mining] - Project/Data/pre-processed/documents.jsonl
  [OK] /content/drive/Shareddrives/[Text Mining] - Project/Data/pre-processed/provisions.jsonl
  [OK] /content/drive/Shareddrives/[Text Mining] - Project/Data/pre-processed/chunks.jsonl
  [OK] /content/drive/Shareddrives/[Text Mining] - Project/Data/pre-processed/edges.jsonl
  [OK] /content/drive/Shareddrives/[Text Mining] - Project/Data/pre-processed/external_stubs.jsonl
Overlay files (optional, still from JSONL after structural load):
  [OK] /content/drive/Shareddrives/[Text Mining] - Project/Data/pre-processed/validi

In [ ]:
# ### HYBRID_GRAPH_INTEGRATION — two-stage hybrid helper + views (US1)


@dataclass
class SeedRetrievalView:
    query: str
    filter_profile: str
    total_candidates: int
    seed_chunks: list[RetrievedChunk]
    seed_chunk_ids: list[str]
    mode_label: str  # vector_only | hybrid_seed


@dataclass
class GraphExpansionView:
    expansion: ExpansionResult | None
    expanded_chunk_ids: list[str]
    added_chunk_ids: list[str]
    resolved_chunks: list[RetrievedChunk]
    warnings: list[str]
    capped: bool
    mechanism_label: str = 'graph_expansion'


@dataclass
class HybridEvidenceContext:
    query: str
    mode: str  # vector_only | hybrid_expanded | graph_guided_prefilter
    seed: SeedRetrievalView
    expansion: GraphExpansionView | None
    evidence_chunks: list[RetrievedChunk]
    document_overlays: dict[str, DocumentOverlay]
    overlay_available: bool
    expansion_added_context: bool
    diagnostics: list[str] = field(default_factory=list)


@dataclass
class ModeComparisonRecord:
    query: str
    vector_only_count: int
    hybrid_count: int
    expansion_ran: bool
    added_context_count: int
    sample_vector_only_ids: list[str]
    sample_hybrid_ids: list[str]
    notes: list[str] = field(default_factory=list)


@dataclass
class GraphGuidedDemoResult:
    start_id: str
    traversal_mode: str
    whitelist_size: int
    empty_filter_warning: bool
    filter_reason: str
    retrieval: RetrievalResult | None


def _hit_to_retrieved_chunk(hit: SearchHit, query: str, filter_profile: str) -> RetrievedChunk:
    """Reuse VectorRetriever conversion so identity fields stay consistent."""
    return retriever._to_retrieved_chunk(hit, query, filter_profile)


def _is_citation_safe_chunk(chunk: RetrievedChunk) -> bool:
    """External stubs / non-citation-safe nodes are never citation-ready (FR-013)."""
    meta = chunk.metadata or {}
    if meta.get('is_external_stub') is True:
        return False
    if meta.get('citation_safe') is False:
        return False
    # Graph external stubs keyed by id_str
    if kg_graph is not None and chunk.id_str and chunk.id_str in getattr(kg_graph, 'external_stubs', {}):
        return False
    if not (chunk.chunk_text or '').strip():
        return False
    return True


def _resolve_chunk_ids(chunk_ids: list[str], query: str, filter_profile: str) -> list[RetrievedChunk]:
    if not chunk_ids:
        return []
    # Preserve order from expansion; scroll may return unordered.
    hits = store.scroll({'chunk_id': {'in': list(chunk_ids)}}, limit=max(len(chunk_ids), 1))
    by_id: dict[str, SearchHit] = {}
    for hit in hits:
        cid = str(hit.payload.get('chunk_id') or hit.point_id)
        by_id[cid] = hit
    ordered: list[RetrievedChunk] = []
    for cid in chunk_ids:
        hit = by_id.get(cid)
        if hit is None:
            continue
        ordered.append(_hit_to_retrieved_chunk(hit, query, filter_profile))
    return ordered


def run_hybrid_retrieve(
    query: str,
    *,
    top_n: int = TOP_N,
    filter_profile: str = FILTER_PROFILE,
    score_threshold: float | None = SCORE_THRESHOLD,
    enable_expansion: bool | None = None,
    max_hop: int | None = None,
    max_context: int | None = None,
) -> HybridEvidenceContext:
    """Vector seed → optional GraphExpansion → overlay join → HybridEvidenceContext."""
    do_expand = ENABLE_HYBRID_EXPANSION if enable_expansion is None else enable_expansion
    max_hop = HYBRID_MAX_HOP if max_hop is None else max_hop
    max_context = HYBRID_MAX_CONTEXT if max_context is None else max_context
    diagnostics: list[str] = []

    # Stage 1: seed vector retrieve (never local expand here)
    seed_result = retriever.retrieve(
        query,
        top_n=top_n,
        filter_profile=filter_profile,
        score_threshold=score_threshold,
        expand_units=False,
    )
    seed_chunks = list(seed_result.chunks)
    seed_ids = [c.chunk_id for c in seed_chunks]
    seed_view = SeedRetrievalView(
        query=query,
        filter_profile=seed_result.filter_profile_used,
        total_candidates=seed_result.total_candidates,
        seed_chunks=seed_chunks,
        seed_chunk_ids=seed_ids,
        mode_label='hybrid_seed' if do_expand else 'vector_only',
    )

    if not do_expand:
        diagnostics.append('Hybrid expansion disabled — returning vector-only seeds.')
        return HybridEvidenceContext(
            query=query,
            mode='vector_only',
            seed=seed_view,
            expansion=None,
            evidence_chunks=seed_chunks,
            document_overlays={},
            overlay_available=graph_load_status.overlays_ready,
            expansion_added_context=False,
            diagnostics=diagnostics,
        )

    require_graph_for_hybrid('hybrid expansion')

    if not seed_ids:
        diagnostics.append('Zero seed hits — skipping graph expansion and recording empty context.')
        empty_expansion = GraphExpansionView(
            expansion=None,
            expanded_chunk_ids=[],
            added_chunk_ids=[],
            resolved_chunks=[],
            warnings=['No seed chunk IDs; expansion skipped.'],
            capped=False,
            mechanism_label='graph_expansion',
        )
        return HybridEvidenceContext(
            query=query,
            mode='hybrid_expanded',
            seed=seed_view,
            expansion=empty_expansion,
            evidence_chunks=[],
            document_overlays={},
            overlay_available=graph_load_status.overlays_ready,
            expansion_added_context=False,
            diagnostics=diagnostics,
        )

    expansion_result = graph_expansion.expand(
        seed_ids,
        max_hop=max_hop,
        max_context=max_context,
    )
    expanded_ids = list(expansion_result.ordered_context_chunks)
    seed_set = set(seed_ids)
    added_ids = [cid for cid in expanded_ids if cid not in seed_set]
    # Prefer expanded order; if expansion returned nothing usable, fall back to seeds
    ordered_ids = expanded_ids or list(seed_ids)
    capped = bool(
        max_context is not None
        and expansion_result.max_context is not None
        and len(expanded_ids) >= int(expansion_result.max_context)
    )
    if capped:
        diagnostics.append(f'Expansion context capped at max_context={max_context}.')

    warnings = list(expansion_result.warnings or ())
    resolved = _resolve_chunk_ids(ordered_ids, query, filter_profile)
    # Keep citation-ready only for generation/display of "evidence"
    citation_ready = [c for c in resolved if _is_citation_safe_chunk(c)]
    dropped = len(resolved) - len(citation_ready)
    if dropped:
        diagnostics.append(f'Excluded {dropped} non-citation-safe/stub/empty chunks from citation-ready evidence.')

    if not added_ids:
        diagnostics.append('Graph expansion ran with zero added neighbors (seeds only) — not a failure.')
    else:
        diagnostics.append(f'Graph expansion added {len(added_ids)} chunk ids beyond seeds.')

    for w in warnings:
        diagnostics.append(f'Expansion warning: {w}')

    expansion_view = GraphExpansionView(
        expansion=expansion_result,
        expanded_chunk_ids=expanded_ids,
        added_chunk_ids=added_ids,
        resolved_chunks=resolved,
        warnings=warnings,
        capped=capped,
        mechanism_label='graph_expansion',
    )

    # Overlay join by id_str (display-only signals; do not mutate payloads)
    involved_ids = {c.id_str for c in citation_ready if c.id_str}
    subset_overlays: dict[str, DocumentOverlay] = {}
    if graph_load_status.overlays_ready and document_overlays:
        subset_overlays = {i: document_overlays[i] for i in involved_ids if i in document_overlays}
        diagnostics.append(f'Overlays attached for {len(subset_overlays)}/{len(involved_ids)} involved documents (as_of={AS_OF_DATE}).')
    else:
        diagnostics.append('Overlays unavailable — structural expansion only; no authoritative currency claims.')

    evidence = citation_ready if citation_ready else list(seed_chunks)
    return HybridEvidenceContext(
        query=query,
        mode='hybrid_expanded',
        seed=seed_view,
        expansion=expansion_view,
        evidence_chunks=evidence,
        document_overlays=subset_overlays,
        overlay_available=graph_load_status.overlays_ready,
        expansion_added_context=bool(added_ids),
        diagnostics=diagnostics,
    )


def chunks_to_display_rows(chunks: list[RetrievedChunk], *, limit: int | None = None) -> list[dict[str, Any]]:
    rows = []
    for rank, chunk in enumerate(chunks[: limit or len(chunks)], start=1):
        rows.append({
            'rank': rank,
            'chunk_id': chunk.chunk_id,
            'parent_unit_id': chunk.parent_unit_id,
            'id_str': chunk.id_str,
            'citation': chunk.citation_anchor or chunk.citation_label,
            'title': chunk.title,
            'validity_group': chunk.validity_group,
            'legal_authority_rank': chunk.legal_authority_rank,
            'vector_score': round(chunk.vector_score, 4),
            'text': (chunk.chunk_text or '')[:400],
        })
    return rows


print('Hybrid helper ready: run_hybrid_retrieve(), require_graph_for_hybrid(), chunks_to_display_rows()')


## 5. Retrieval helper

In [ ]:
def search(
    query: str,
    top_n: int = TOP_N,
    filter_profile: str = FILTER_PROFILE,
    score_threshold: float | None = SCORE_THRESHOLD,
    expand_units: bool | None = None,
    graph_guided_filter=None,
    use_hybrid_retriever: bool = False,
):
    """Run VectorRetriever and return (display_rows, RetrievalResult).

    filter_profile: 'current_law' | 'broad' | 'historical' | 'graph_guided' (via graph_guided_filter)

    Expansion labeling:
    - expand_units=True with graph_expansion wired → mechanism is graph expansion (module path)
    - expand_units=True without graph_expansion → local_expand_units (payload same-provision)
    Prefer the two-stage hybrid helper for seed vs expanded diagnostics.
    """
    active = hybrid_retriever if (use_hybrid_retriever and hybrid_retriever is not None) else retriever

    if filter_profile == 'graph_guided' and graph_guided_filter is None:
        print(
            'graph_guided requested without GraphGuidedFilter. '
            'Use the optional graph-guided pre-filter demo, or pass graph_guided_filter=... '
            'Falling back to broad (pure vector).'
        )
        filter_profile = 'broad'

    search_t0 = time.perf_counter()
    result = active.retrieve(
        query,
        top_n=top_n,
        filter_profile=filter_profile if graph_guided_filter is None else 'graph_guided',
        score_threshold=score_threshold,
        expand_units=LOCAL_EXPAND_UNITS if expand_units is None else expand_units,
        graph_guided_filter=graph_guided_filter,
    )
    print(f'Retrieval completed in {time.perf_counter() - search_t0:.2f}s')
    rows = []
    for rank, chunk in enumerate(result.chunks, start=1):
        rows.append({
            'rank': rank,
            'chunk_id': chunk.chunk_id,
            'id_str': chunk.id_str,
            'citation': chunk.citation_anchor or chunk.citation_label,
            'title': chunk.title,
            'unit_type': chunk.unit_type,
            'validity_group': chunk.validity_group,
            'parent_unit_id': chunk.parent_unit_id,
            'vector_score': round(chunk.vector_score, 4),
            'rerank_score': round(chunk.rerank_score, 4),
            'text': chunk.chunk_text[:700],
        })
    return rows, result


def show_results(rows):
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(json.dumps(row, ensure_ascii=False, indent=2))


## 5.1 Benchmark helper

In [ ]:
def benchmark_search(query: str, repeats: int = 3, filter_profile: str = FILTER_PROFILE):
    timings = []
    for i in range(repeats):
        t0 = time.perf_counter()
        rows, result = search(query, top_n=TOP_N, filter_profile=filter_profile)
        elapsed = time.perf_counter() - t0
        timings.append(elapsed)
        print(f'Run {i + 1}/{repeats}: {elapsed:.2f}s, returned={len(result.chunks)}, candidates={result.total_candidates}')
    avg = sum(timings) / len(timings)
    print(f'Average retrieval time over {repeats} runs: {avg:.2f}s')
    return timings


## 6. Run a query

In [ ]:
query_t0 = time.perf_counter()
query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'

# Primary run under default FILTER_PROFILE (pure vector / non-graph)
rows, result = search(query, top_n=10, filter_profile=FILTER_PROFILE)
print('Filter profile used:', result.filter_profile_used)
print('Total candidates:', result.total_candidates)
print('Empty filter warning:', result.empty_filter_warning)
show_results(rows)
print(f'Query phase completed in {time.perf_counter() - query_t0:.2f}s')

# --- Filter-profile comparison (non-graph profiles remain fully usable without graph) ---
print('\n=== Filter profile comparison (same query) ===')
for profile in ('current_law', 'broad', 'historical'):
    _, r = search(query, top_n=TOP_N, filter_profile=profile)
    print(
        f'  {profile:12s} | candidates={r.total_candidates:4d} | '
        f'returned={len(r.chunks):2d} | empty_filter_warning={r.empty_filter_warning}'
    )

print(
    '  graph_guided  | secondary path — see optional demo cell '
    f'(ENABLE_GRAPH_GUIDED_PREFILTER_DEMO={ENABLE_GRAPH_GUIDED_PREFILTER_DEMO})'
)

# --- Local same-provision expansion demo (local_expand_units mechanism) ---
print('\n=== Local expansion demo (local_expand_units=True vs False) ===')
print('Mechanism label: local_expand_units (payload same-provision window; not graph_expansion)')
_, r_no = search(query, top_n=5, filter_profile='broad', expand_units=False)
_, r_yes = search(query, top_n=5, filter_profile='broad', expand_units=True)
print(f'  local_expand_units=False → {len(r_no.chunks)} chunks')
print(f'  local_expand_units=True  → {len(r_yes.chunks)} chunks')
parent_ids = {c.parent_unit_id for c in r_yes.chunks if c.parent_unit_id}
print(f'  unique parent_unit_id among expanded results: {len(parent_ids)}')


## 6.1 Hybrid expansion demo, diagnostics, and comparison

Seed vs graph-expanded evidence, overlay signals, vector-only vs hybrid comparison, and optional graph-guided pre-filter (secondary).

Labels: **graph_expansion** is distinct from **local_expand_units**.


In [ ]:
# ### HYBRID_GRAPH_INTEGRATION — diagnostics, comparison, demos (US2/US3/US4)

hybrid_demo_query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'


def show_hybrid_diagnostics(ctx: HybridEvidenceContext) -> None:
    """Seed vs expanded counts, identity samples, overlays, warnings (US2)."""
    print('=== Hybrid diagnostics ===')
    print('Query:', ctx.query)
    print('Mode:', ctx.mode)
    print('Mechanism: graph_expansion (not local_expand_units)')
    print(f'Seed count: {len(ctx.seed.seed_chunks)} | seed candidates: {ctx.seed.total_candidates}')
    if ctx.expansion is None:
        print('Expansion: not run')
    else:
        print(
            f'Expanded ids: {len(ctx.expansion.expanded_chunk_ids)} | '
            f'added: {len(ctx.expansion.added_chunk_ids)} | '
            f'capped: {ctx.expansion.capped} | '
            f'mechanism_label: {ctx.expansion.mechanism_label}'
        )
        if ctx.expansion.warnings:
            print('Expansion warnings:')
            for w in ctx.expansion.warnings:
                print(' -', w)
    print(f'Evidence (citation-ready) count: {len(ctx.evidence_chunks)}')
    print('Sample identity chain (chunk_id → parent_unit_id → id_str):')
    show_results(chunks_to_display_rows(ctx.evidence_chunks, limit=8))

    print('\n--- Overlay diagnostics ---')
    if ctx.overlay_available and ctx.document_overlays:
        print(f'Overlay coverage for involved docs: {len(ctx.document_overlays)}')
        sample_id, sample_ov = next(iter(ctx.document_overlays.items()))
        print(f'Sample document id_str={sample_id}')
        print(f'  currency_status: {sample_ov.currency_status}')
        print(f'  currency_status_as_of: {sample_ov.currency_status_as_of}')
        print(f'  legal_authority_rank: {sample_ov.legal_authority_rank}')
        print(f'  authority_rank_source: {sample_ov.authority_rank_source}')
    else:
        print('Overlays unavailable or none matched involved documents — no authoritative currency/authority claims.')

    if ctx.diagnostics:
        print('\nStage notes:')
        for note in ctx.diagnostics:
            print(' -', note)


def compare_vector_vs_hybrid(
    query: str,
    *,
    top_n: int = TOP_N,
    filter_profile: str = FILTER_PROFILE,
) -> ModeComparisonRecord:
    """Same query under vector_only vs hybrid_expanded (US3 / FR-011)."""
    notes: list[str] = []
    # Vector-only
    vo_result = retriever.retrieve(
        query,
        top_n=top_n,
        filter_profile=filter_profile,
        score_threshold=SCORE_THRESHOLD,
        expand_units=False,
    )
    vo_ids = [c.chunk_id for c in vo_result.chunks]

    expansion_ran = False
    added = 0
    hybrid_ids: list[str] = []
    hybrid_count = 0
    try:
        require_graph_for_hybrid('hybrid side of mode comparison')
        hctx = run_hybrid_retrieve(query, top_n=top_n, filter_profile=filter_profile, enable_expansion=True)
        expansion_ran = hctx.expansion is not None and hctx.mode == 'hybrid_expanded'
        added = len(hctx.expansion.added_chunk_ids) if hctx.expansion else 0
        hybrid_ids = [c.chunk_id for c in hctx.evidence_chunks]
        hybrid_count = len(hctx.evidence_chunks)
        if expansion_ran and added == 0:
            notes.append('Expansion ran but added nothing beyond seeds.')
        elif expansion_ran:
            notes.append(f'Expansion added {added} chunk ids.')
        notes.extend(hctx.diagnostics[:5])
    except RuntimeError as exc:
        notes.append(f'Hybrid unavailable: {exc}')
        hybrid_count = -1

    record = ModeComparisonRecord(
        query=query,
        vector_only_count=len(vo_result.chunks),
        hybrid_count=hybrid_count,
        expansion_ran=expansion_ran,
        added_context_count=added,
        sample_vector_only_ids=vo_ids[:5],
        sample_hybrid_ids=hybrid_ids[:5],
        notes=notes,
    )
    print('=== Mode comparison: vector_only vs hybrid_expanded ===')
    print('Query:', query)
    print(f'  vector_only     count={record.vector_only_count} sample_ids={record.sample_vector_only_ids}')
    print(f'  hybrid_expanded count={record.hybrid_count} expansion_ran={record.expansion_ran} added={record.added_context_count}')
    print(f'  sample_hybrid_ids={record.sample_hybrid_ids}')
    for n in record.notes:
        print('  note:', n)
    return record


def run_graph_guided_prefilter_demo(
    query: str,
    *,
    start_id: str | None = None,
    top_n: int = TOP_N,
    filter_profile: str = 'current_law',
) -> GraphGuidedDemoResult:
    """Secondary whitelist-before-search path (US4 / FR-020). Not the default ask() path."""
    require_graph_for_hybrid('graph-guided pre-filter demo')
    assert kg_facade is not None and kg_graph is not None

    sid = (start_id or GRAPH_GUIDED_START_ID or '').strip()
    if not sid:
        # Derive from a seed hit when possible
        seed = retriever.retrieve(query, top_n=max(1, top_n), filter_profile=FILTER_PROFILE, expand_units=False)
        if seed.chunks and seed.chunks[0].id_str:
            sid = seed.chunks[0].id_str
            print(f'Graph-guided start id_str taken from first seed hit: {sid}')
        else:
            # Fall back to first document with a verified edge
            for edge in kg_graph.verified_document_edges:
                if edge.src_id in kg_graph.documents:
                    sid = edge.src_id
                    print(f'Graph-guided start id_str taken from verified edge src: {sid}')
                    break
    if not sid:
        raise RuntimeError('No start id_str available for graph-guided pre-filter demo.')

    traversal = kg_facade.traverse(
        kg_graph,
        start_id=sid,
        mode=GRAPH_GUIDED_TRAVERSAL_MODE,  # type: ignore[arg-type]
        max_depth=GRAPH_GUIDED_MAX_DEPTH,
    )
    overlays = document_overlays if graph_load_status.overlays_ready else {}
    guided = kg_facade.build_graph_guided_filter(
        graph=kg_graph,
        traversal=traversal,
        overlays=overlays,
        filter_profile=filter_profile,  # type: ignore[arg-type]
        constraints=QueryConstraints(validity_groups=('active', 'partial', 'future')),
    )
    print('=== Graph-guided pre-filter demo (SECONDARY path) ===')
    print('start_id:', sid)
    print('traversal_mode:', GRAPH_GUIDED_TRAVERSAL_MODE)
    print('whitelist size:', len(guided.id_strs))
    print('empty_filter_warning:', guided.empty_filter_warning)
    print('filter_profile:', guided.filter_profile)
    print('filter reason:', getattr(guided, 'reason', '') or '(none)')

    retrieval = None
    if guided.empty_filter_warning or not guided.id_strs:
        print(
            'EMPTY whitelist — not searching full corpus under a graph-guided label. '
            'empty_filter_warning stays True; no unfiltered hits returned as graph-guided.'
        )
        retrieval = RetrievalResult([], 0, 'graph_guided', empty_filter_warning=True)
    else:
        retrieval = retriever.retrieve(
            query,
            top_n=top_n,
            graph_guided_filter=guided,
            expand_units=False,
        )
        print('graph-guided retrieval returned:', len(retrieval.chunks), 'chunks')
        print('empty_filter_warning on result:', retrieval.empty_filter_warning)
        show_results(chunks_to_display_rows(retrieval.chunks, limit=8))

    return GraphGuidedDemoResult(
        start_id=sid,
        traversal_mode=str(GRAPH_GUIDED_TRAVERSAL_MODE),
        whitelist_size=len(guided.id_strs),
        empty_filter_warning=bool(guided.empty_filter_warning or not guided.id_strs),
        filter_reason=str(getattr(guided, 'reason', '') or guided.filter_profile),
        retrieval=retrieval,
    )


# --- Demo runs (safe when graph missing: hybrid calls fail clearly) ---
print('Running hybrid demo query when enabled...')
hybrid_ctx = None
comparison_record = None
graph_guided_demo = None

if ENABLE_HYBRID_EXPANSION and graph_load_status.structural_ready:
    hybrid_ctx = run_hybrid_retrieve(hybrid_demo_query, top_n=TOP_N, filter_profile=FILTER_PROFILE)
    show_hybrid_diagnostics(hybrid_ctx)
    comparison_record = compare_vector_vs_hybrid(hybrid_demo_query)
elif ENABLE_HYBRID_EXPANSION and not graph_load_status.structural_ready:
    print('Hybrid enabled but graph unavailable — demonstrating FR-015 clear failure:')
    try:
        require_graph_for_hybrid('hybrid demo')
    except RuntimeError as exc:
        print('Expected failure:', exc)
    print('Pure vector search still works:')
    rows_v, res_v = search(hybrid_demo_query, top_n=5, filter_profile=FILTER_PROFILE, expand_units=False)
    print('vector_only returned:', len(res_v.chunks))
else:
    print('ENABLE_HYBRID_EXPANSION=False — skip hybrid demo. Vector-only path remains default.')

if ENABLE_GRAPH_GUIDED_PREFILTER_DEMO:
    if graph_load_status.structural_ready:
        graph_guided_demo = run_graph_guided_prefilter_demo(hybrid_demo_query)
    else:
        print('Graph-guided demo enabled but graph unavailable — skipping with explicit message.')
else:
    print('ENABLE_GRAPH_GUIDED_PREFILTER_DEMO=False — secondary pre-filter demo not run.')


## 7. Optional: inspect one full chunk

In [ ]:
if result.chunks:
    chunk = result.chunks[0]
    print('chunk_id:', chunk.chunk_id)
    print('citation:', chunk.citation_anchor or chunk.citation_label)
    print('title:', chunk.title)
    print('scores:', {'vector': chunk.vector_score, 'rerank': chunk.rerank_score})
    print('--- text ---')
    print(chunk.chunk_text)
    print('--- metadata keys ---')
    print(sorted(chunk.metadata.keys()))


## 8. Configure the answer generator (OpenAI-compatible API)

Set `LLM_BASE_URL`, `LLM_API_KEY`, and `LLM_MODEL_NAME` via environment variables so credentials never end up hardcoded in this notebook. Any OpenAI-compatible chat completions endpoint works (OpenAI, Azure OpenAI, vLLM, Together, OpenRouter, etc.).

```bash
export LLM_BASE_URL="https://api.your-provider.com/v1"
export LLM_API_KEY="..."
export LLM_MODEL_NAME="gpt-4o-mini"
```

If these are not set, the notebook still runs in retrieval-only mode; generation cells will skip cleanly.

**Security (FR-018):** only a masked key is printed. Raw keys are never logged.


In [ ]:
from generation.reasoning_client import GeneratorConfig

generator_config = GeneratorConfig(
    base_url=os.environ.get('LLM_BASE_URL', '').strip(),
    api_key=os.environ.get('LLM_API_KEY', '').strip(),
    model_name=os.environ.get('LLM_MODEL_NAME', '').strip(),
)

# Back-compat aliases used by older cells / mental model
BASE_URL = generator_config.base_url
API_KEY = generator_config.api_key
MODEL_NAME = generator_config.model_name

if not generator_config.is_complete():
    print('Generator not fully configured. Set LLM_BASE_URL, LLM_API_KEY, LLM_MODEL_NAME env vars to enable answer generation.')
else:
    print('Generator configured:')
    print('  BASE_URL:', BASE_URL)
    print('  MODEL_NAME:', MODEL_NAME)
    print('  API_KEY:', generator_config.masked_key())


## 9. Generator client and answer-generation helper

Uses the extracted module `generation.reasoning_client`:
- `GeneratorClient.generate` → `RawGenerationResponse`
- `parse_generation_response` — three shapes: dedicated reasoning field, `<think>...</think>` block, or `not_returned`
- `generate_answer` → `GenerationOutcome` (skip empty context / error / parsed)


In [ ]:
from generation.reasoning_client import (
    ANSWER_PROMPT,
    GenerationOutcome,
    GeneratorClient,
    ParsedAnswer,
    RawGenerationResponse,
    format_context_for_prompt,
    generate_answer as _generate_answer_outcome,
    parse_generation_response,
)

generator: GeneratorClient | None = None
if generator_config.is_complete():
    generator = GeneratorClient(
        base_url=generator_config.base_url,
        api_key=generator_config.api_key,
        model=generator_config.model_name,
    )
    print('Generator client ready.')
    print('ANSWER_PROMPT includes reasoning instruction:', 'reasoning' in ANSWER_PROMPT.lower() or 'suy luận' in ANSWER_PROMPT.lower())
else:
    print('Generator client not created (missing config). Retrieval-only mode.')


def generate_answer(query: str, chunks, *, qa_id: str | None = None) -> GenerationOutcome:
    """Notebook wrapper: returns GenerationOutcome (never raises for empty context)."""
    gen_t0 = time.perf_counter()
    outcome = _generate_answer_outcome(generator, query, chunks, qa_id=qa_id)
    print(f'Generation completed in {time.perf_counter() - gen_t0:.2f}s')
    return outcome


def display_generation_outcome(outcome: GenerationOutcome) -> None:
    """Print answer and reasoning as two distinct sections (FR-020/FR-021)."""
    if outcome.skipped_empty_context:
        print('Skipped generation: empty retrieved context.')
        return
    if outcome.error:
        print('--- Generation error ---')
        print(outcome.error)
        return
    parsed = outcome.parsed
    assert parsed is not None
    print('\n--- Answer ---')
    print(parsed.answer or '(empty)')
    print('\n--- Reasoning ---')
    if parsed.reasoning_available and parsed.reasoning:
        print(f'(source={parsed.reasoning_source})')
        print(parsed.reasoning)
    else:
        print('not returned by this model')


## 10. Full RAG pipeline: retrieve + generate

Ad hoc `ask()` runs retrieval then generation. Final answer and model reasoning are shown as **two distinct sections**. If the model does not return reasoning, the notebook prints `not returned by this model` rather than inventing text.


In [ ]:
def ask(
    query: str,
    top_n: int = TOP_N,
    filter_profile: str = FILTER_PROFILE,
    score_threshold: float | None = SCORE_THRESHOLD,
    *,
    use_hybrid: bool | None = None,
    use_hybrid_evidence: bool | None = None,
):
    """Full pipeline: vector seed → (optional graph expand/overlays) → generate.

    Default demonstration path uses hybrid expanded evidence when
    ENABLE_HYBRID_EXPANSION and the graph is loaded (FR-009 / FR-022).
    Hybrid requested while graph unavailable fails clearly (FR-015).
    """
    hybrid = ENABLE_HYBRID_EXPANSION if use_hybrid is None else use_hybrid
    hybrid_for_gen = USE_HYBRID_EVIDENCE_FOR_GENERATION if use_hybrid_evidence is None else use_hybrid_evidence

    hybrid_ctx = None
    result = None
    evidence_chunks = []

    if hybrid:
        require_graph_for_hybrid('hybrid ask() full pipeline')
        hybrid_ctx = run_hybrid_retrieve(
            query,
            top_n=top_n,
            filter_profile=filter_profile,
            score_threshold=score_threshold,
            enable_expansion=True,
        )
        print('Mode:', hybrid_ctx.mode)
        print('Seed candidates:', hybrid_ctx.seed.total_candidates)
        print('Evidence chunks:', len(hybrid_ctx.evidence_chunks))
        print('Expansion added context:', hybrid_ctx.expansion_added_context)
        if hybrid_ctx.diagnostics:
            print('Diagnostics:')
            for note in hybrid_ctx.diagnostics:
                print(' -', note)
        show_results(chunks_to_display_rows(hybrid_ctx.evidence_chunks))
        evidence_chunks = list(hybrid_ctx.evidence_chunks)
        # Build a RetrievalResult-like object for callers that expect .chunks
        from retrieval.schema import RetrievalResult
        result = RetrievalResult(
            chunks=evidence_chunks,
            total_candidates=hybrid_ctx.seed.total_candidates,
            filter_profile_used=filter_profile,
            empty_filter_warning=False,
        )
    else:
        rows, result = search(
            query,
            top_n=top_n,
            filter_profile=filter_profile,
            score_threshold=score_threshold,
            expand_units=False,
        )
        print('Mode: vector_only')
        print('Filter profile used:', result.filter_profile_used)
        print('Total candidates:', result.total_candidates)
        print('Empty filter warning:', result.empty_filter_warning)
        show_results(rows)
        evidence_chunks = list(result.chunks)

    usable = [c for c in evidence_chunks if (c.chunk_text or '').strip()]
    if not usable:
        print('No usable evidence text after retrieval/expansion; skipping generation (empty context).')
        outcome = GenerationOutcome(qa_id=None, parsed=None, skipped_empty_context=True, error=None)
        return {
            'query': query,
            'outcome': outcome,
            'result': result,
            'hybrid': hybrid_ctx,
            'mode': hybrid_ctx.mode if hybrid_ctx else 'vector_only',
        }

    if generator is None:
        print('Generator not configured; returning retrieval/expansion-only result.')
        return {
            'query': query,
            'outcome': None,
            'result': result,
            'hybrid': hybrid_ctx,
            'mode': hybrid_ctx.mode if hybrid_ctx else 'vector_only',
        }

    if hybrid and hybrid_for_gen:
        gen_chunks = usable
        print('Generation uses hybrid expanded evidence (USE_HYBRID_EVIDENCE_FOR_GENERATION=True).')
    elif hybrid and not hybrid_for_gen:
        # Prefer seed-only evidence when hybrid retrieval ran but generation should not use expansion.
        seed_only = list(hybrid_ctx.seed.seed_chunks) if hybrid_ctx is not None else usable
        gen_chunks = [c for c in seed_only if (c.chunk_text or '').strip()] or usable
        print('Generation uses seed-only evidence (USE_HYBRID_EVIDENCE_FOR_GENERATION=False).')
    else:
        gen_chunks = usable
    outcome = generate_answer(query, gen_chunks)
    display_generation_outcome(outcome)

    print('\n--- Citations used (citation-ready evidence only) ---')
    for rank, chunk in enumerate(gen_chunks, start=1):
        print(
            f'[{rank}] {chunk.citation_anchor or chunk.citation_label} - {chunk.title} '
            f'| chunk_id={chunk.chunk_id} → parent_unit_id={chunk.parent_unit_id} → id_str={chunk.id_str}'
        )
    return {
        'query': query,
        'outcome': outcome,
        'result': result,
        'hybrid': hybrid_ctx,
        'mode': hybrid_ctx.mode if hybrid_ctx else 'vector_only',
    }


pipeline_query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'
# Uncomment when ready (generator optional — hybrid retrieval still completes without credentials):
# pipeline_output = ask(pipeline_query, top_n=10, filter_profile='broad')
print('ask() defined. Default path: vector seed → graph expand/overlays → generate when hybrid enabled.')
print('Call: pipeline_output = ask(pipeline_query, top_n=10, filter_profile="broad")')
print('Vector-only: pipeline_output = ask(pipeline_query, use_hybrid=False)')


## 11. Optional: run the full pipeline over a benchmark sample

`run_benchmark_sample` scores retrieval hit-rate against `data/benchmark/qa_final.jsonl` and, when a generator is configured, records per-question `GenerationOutcome` (answer, reasoning source, errors). A single generation failure does **not** stop the loop (FR-023 / SC-008).

This is a demo/validation path — not a full LLM-as-judge evaluation. Use `scripts/evaluate_e2e.py` for judged scoring (FR-024).


In [ ]:
import random


def run_benchmark_sample(
    qa_path: Path | None = None,
    sample_size: int = BENCHMARK_SAMPLE_SIZE,
    filter_profile: str = FILTER_PROFILE,
    seed: int = 42,
    run_generation: bool = True,
):
    """Run retrieval (+ generation, if configured) over a random sample of qa_final.jsonl.

    Reports per-question retrieval hit (whether any ground-truth chunk/provision/document id
    appears among the retrieved results) plus an aggregate hit rate and average latency.
    Unanswerable questions (empty ground_truth) are excluded from the hit-rate denominator.

    Generation reuses already-retrieved chunks (no re-query). Failures are recorded per
    question and do not abort the loop.
    """
    qa_path = qa_path or (PROJECT_ROOT / 'data' / 'benchmark' / 'qa_final.jsonl')
    if not qa_path.exists():
        raise FileNotFoundError(f'Benchmark file not found at {qa_path}')

    with qa_path.open('r', encoding='utf-8') as f:
        all_cases = [json.loads(line) for line in f if line.strip()]

    rng = random.Random(seed)
    sample = rng.sample(all_cases, min(sample_size, len(all_cases)))

    records = []
    latencies = []
    hits = 0
    scored = 0
    gen_errors = 0
    gen_skipped = 0
    gen_ok = 0

    for qa in sample:
        question = qa.get('question') or ''
        ground_truth = qa.get('ground_truth') or {}
        gt_ids = (
            set(ground_truth.get('chunk_ids') or [])
            | set(ground_truth.get('provision_ids') or [])
            | set(ground_truth.get('document_ids') or [])
        )
        is_unanswerable = qa.get('answer_type') == 'unanswerable' or not gt_ids

        t0 = time.perf_counter()
        _, result = search(question, top_n=TOP_N, filter_profile=filter_profile)
        elapsed = time.perf_counter() - t0
        latencies.append(elapsed)

        retrieved_ids = set()
        for chunk in result.chunks:
            retrieved_ids.update({chunk.chunk_id, chunk.parent_unit_id, chunk.id_str})

        hit = bool(gt_ids & retrieved_ids)
        if not is_unanswerable:
            scored += 1
            if hit:
                hits += 1

        outcome: GenerationOutcome | None = None
        if run_generation and generator is not None:
            # generate_answer never raises for empty context; API errors become outcome.error
            outcome = _generate_answer_outcome(
                generator,
                question,
                result.chunks,
                qa_id=qa.get('qa_id'),
            )
            if outcome.skipped_empty_context:
                gen_skipped += 1
            elif outcome.error:
                gen_errors += 1
            elif outcome.parsed is not None:
                gen_ok += 1

        parsed = outcome.parsed if outcome else None
        records.append({
            'qa_id': qa.get('qa_id'),
            'question': question,
            'answer_type': qa.get('answer_type'),
            'category': qa.get('category'),
            'is_unanswerable': is_unanswerable,
            'retrieval_hit': hit,
            'total_candidates': result.total_candidates,
            'latency_s': round(elapsed, 3),
            'generated_answer': parsed.answer if parsed else None,
            'reasoning': parsed.reasoning if parsed else None,
            'reasoning_source': parsed.reasoning_source if parsed else None,
            'reasoning_available': parsed.reasoning_available if parsed else False,
            'generation_error': outcome.error if outcome else None,
            'skipped_empty_context': outcome.skipped_empty_context if outcome else False,
        })

    hit_rate = hits / scored if scored else float('nan')
    avg_latency = sum(latencies) / len(latencies) if latencies else float('nan')

    print(f'Sampled {len(sample)} questions ({scored} scored, {len(sample) - scored} unanswerable excluded)')
    print(f'Hit rate: {hit_rate:.2%}' if scored else 'Hit rate: n/a (no scored questions)')
    print(f'Average retrieval latency: {avg_latency:.3f}s')
    if run_generation and generator is not None:
        print(f'Generation: ok={gen_ok}, skipped_empty={gen_skipped}, errors={gen_errors}')

    return {
        'records': records,
        'hit_rate': hit_rate,
        'avg_latency_s': avg_latency,
        'sample_size': len(sample),
        'scored': scored,
        'generation_ok': gen_ok,
        'generation_errors': gen_errors,
        'generation_skipped': gen_skipped,
    }


# Uncomment to run (generation requires LLM_* env vars; retrieval-only works without them):
# benchmark_summary = run_benchmark_sample(sample_size=BENCHMARK_SAMPLE_SIZE, run_generation=True)
print('run_benchmark_sample() defined. Example: benchmark_summary = run_benchmark_sample(sample_size=10)')
